# LCCA-v9 CUDA-parity item-cluster bootstrap

This notebook **does not retrain** any model. It loads the nine archived V9
checkpoints, re-scores the 81-pair O/A/B contrast set on a Colab CUDA GPU, and
first verifies that all nine recalculated summaries exactly match the archived
`metrics.json`. If any value differs, execution stops before inference claims
are written.

Only after the parity gate passes does the notebook save raw item-level
predictions and run 100,000 paired item-cluster bootstrap replicates, a two-way
item/seed sensitivity bootstrap, and item-majority McNemar analysis.

Expected input ZIP: `lcca_v9_reproduction_checkpoints_contrast_artifacts.zip`.
The base model is read from Google Drive; there is no training and no runtime
deletion command.

In [ ]:
# 1) Mount Drive first so permission is granted before long-running cells.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Install exact analysis dependencies without changing Colab's pandas pin.
!pip -q install "transformers==4.57.6" "huggingface-hub==0.36.2" "pandas==2.2.3" sentencepiece protobuf

import gc, hashlib, importlib.util, json, math, os, random, shutil, statistics, zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required. In Colab select Runtime > Change runtime type > GPU.')

DEVICE = torch.device('cuda')
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA autocast dtype:', AMP_DTYPE)

DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/Colab Notebooks/NLP Modeller/deberta-v3-large')
LOCAL_MODEL_DIR = Path('/content/models/deberta-v3-large')
WORK_DIR = Path('/content/lcca_v9_clustered_bootstrap')
ARTIFACT_DIR = WORK_DIR / 'input_artifact'
RESULT_DIR = WORK_DIR / 'results'
for p in (WORK_DIR, ARTIFACT_DIR, RESULT_DIR):
    p.mkdir(parents=True, exist_ok=True)

required_model_files = ['config.json', 'spm.model', 'tokenizer_config.json']
missing = [name for name in required_model_files if not (DRIVE_MODEL_DIR / name).is_file()]
has_weights = any((DRIVE_MODEL_DIR / name).is_file() for name in ('model.safetensors', 'pytorch_model.bin'))
if missing or not has_weights:
    raise FileNotFoundError(f'Incomplete model folder: {DRIVE_MODEL_DIR}; missing={missing}; weights={has_weights}')

shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR, dirs_exist_ok=True)
print('Base model copied locally.')

In [ ]:
# 3) Locate or upload the completed V9 checkpoint artifact ZIP.
from google.colab import files

candidates = list(Path('/content').glob('lcca_v9_reproduction_checkpoints_contrast_artifacts*.zip'))
candidates += list(Path('/content/drive/MyDrive').rglob('lcca_v9_reproduction_checkpoints_contrast_artifacts*.zip'))
if candidates:
    ARTIFACT_ZIP = candidates[0]
else:
    print('Select lcca_v9_reproduction_checkpoints_contrast_artifacts.zip')
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError('No artifact ZIP uploaded.')
    ARTIFACT_ZIP = Path('/content') / next(iter(uploaded))

with zipfile.ZipFile(ARTIFACT_ZIP) as zf:
    bad = zf.testzip()
    if bad:
        raise RuntimeError(f'Corrupt member in ZIP: {bad}')
    zf.extractall(ARTIFACT_DIR)

required = ['metrics.json', 'contrast_set.jsonl', 'score_contrast_set.py', 'checkpoint_manifest.csv']
for name in required:
    if not (ARTIFACT_DIR / name).is_file():
        raise FileNotFoundError(ARTIFACT_DIR / name)

manifest = pd.read_csv(ARTIFACT_DIR / 'checkpoint_manifest.csv')
for row in manifest.itertuples(index=False):
    path = ARTIFACT_DIR / 'checkpoints' / row.filename
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if path.stat().st_size != row.bytes or actual != row.sha256:
        raise RuntimeError(f'Checkpoint manifest mismatch: {row.filename}')
print('ZIP and all nine checkpoint hashes verified:', ARTIFACT_ZIP)

In [ ]:
# 4) Exact V9 scorer definitions and frozen feature cache.
SEEDS = (13, 42, 71)
ARMS = ('o', 'a', 'b')
OPTIONS = ('yes', 'no')
LATENT_RANK = 256
LATENT_HEADS = 8
MLP_BOTTLENECK = 512
MAX_LENGTH = 512
EVAL_BATCH_SIZE = 4

class GatedMLPAdapter(nn.Module):
    def __init__(self, hidden_size, bottleneck, dropout=0.1):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck, bias=False)
        self.up = nn.Linear(bottleneck, hidden_size, bias=False)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.alpha_logit = nn.Parameter(torch.tensor(math.log(0.1 / 0.9), dtype=torch.float32))
    @property
    def alpha(self): return torch.sigmoid(self.alpha_logit)
    def forward(self, hidden, attention_mask=None, enabled=True):
        if not enabled: return hidden
        z = self.up(F.gelu(self.down(hidden)))
        return self.norm(hidden + self.alpha * self.dropout(z))

class LatentRelationAdapter(nn.Module):
    def __init__(self, hidden_size, rank, heads, dropout=0.1):
        super().__init__()
        assert rank % heads == 0
        self.rank, self.heads, self.head_dim = rank, heads, rank // heads
        self.q = nn.Linear(hidden_size, rank, bias=False)
        self.k = nn.Linear(hidden_size, rank, bias=False)
        self.v = nn.Linear(hidden_size, rank, bias=False)
        self.out = nn.Linear(rank, hidden_size, bias=False)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.alpha_logit = nn.Parameter(torch.tensor(math.log(0.1 / 0.9), dtype=torch.float32))
    @property
    def alpha(self): return torch.sigmoid(self.alpha_logit)
    def forward(self, hidden, attention_mask, enabled=True):
        if not enabled: return hidden
        bsz, seq_len, _ = hidden.shape
        def split(x): return x.view(bsz, seq_len, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = split(self.q(hidden)), split(self.k(hidden)), split(self.v(hidden))
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(attention_mask[:, None, None, :] == 0, torch.finfo(scores.dtype).min)
        attn = torch.softmax(scores, dim=-1)
        z = torch.matmul(attn, v).transpose(1, 2).contiguous().view(bsz, seq_len, self.rank)
        return self.norm(hidden + self.alpha * self.dropout(self.out(z)))

class FrozenFeatureScorer(nn.Module):
    def __init__(self, mode, hidden_size):
        super().__init__()
        if mode == 'head_only': self.adapter = None
        elif mode == 'mlp_adapter': self.adapter = GatedMLPAdapter(hidden_size, MLP_BOTTLENECK)
        elif mode == 'lcca_adapter': self.adapter = LatentRelationAdapter(hidden_size, LATENT_RANK, LATENT_HEADS)
        else: raise ValueError(mode)
        self.dropout = nn.Dropout(0.15)
        self.scorer = nn.Linear(hidden_size, 1)
    def forward(self, hidden, attention_mask, adapter_enabled=True):
        if self.adapter is not None:
            hidden = self.adapter(hidden, attention_mask, enabled=adapter_enabled)
        return self.scorer(self.dropout(hidden[:, 0]))

def safe_load(path):
    try: return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError: return torch.load(path, map_location='cpu')

contrast_rows = [json.loads(x) for x in (ARTIFACT_DIR / 'contrast_set.jsonl').read_text().splitlines() if x.strip()]
if len(contrast_rows) != 81: raise ValueError(f'Expected 81 pairs, found {len(contrast_rows)}')

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
encoder = AutoModel.from_pretrained(LOCAL_MODEL_DIR).to(DEVICE).eval()
for p in encoder.parameters(): p.requires_grad = False
hidden_size = int(encoder.config.hidden_size)

unique_inputs, seen = [], set()
for row in contrast_rows:
    for arm in ARMS:
        for option in OPTIONS:
            key = (row[f'passage_{arm}'], row['question'], option)
            if key not in seen:
                seen.add(key); unique_inputs.append(key)

feature_cache = {}
for start in range(0, len(unique_inputs), EVAL_BATCH_SIZE):
    keys = unique_inputs[start:start + EVAL_BATCH_SIZE]
    first = [f'question: {q} answer: {o}' for _, q, o in keys]
    second = [p for p, _, _ in keys]
    tokens = tokenizer(first, second, truncation='only_second', max_length=MAX_LENGTH,
                       padding=True, return_tensors='pt')
    ids, mask = tokens['input_ids'].to(DEVICE), tokens['attention_mask'].to(DEVICE)
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=AMP_DTYPE):
        hidden = encoder(input_ids=ids, attention_mask=mask).last_hidden_state
    for i, key in enumerate(keys):
        length = int(mask[i].sum())
        feature_cache[key] = (hidden[i:i+1, :length].cpu().clone(), mask[i:i+1, :length].cpu().clone())
    if start == 0 or start + len(keys) == len(unique_inputs) or (start + len(keys)) % 100 == 0:
        print(f'Frozen features: {start + len(keys)}/{len(unique_inputs)}')

del encoder
gc.collect(); torch.cuda.empty_cache()
print('Frozen feature cache complete:', len(feature_cache))

In [ ]:
# 5) Score all arms, enforce exact nine-run parity, then save raw predictions.
spec = importlib.util.spec_from_file_location('score_contrast_set_archived', ARTIFACT_DIR / 'score_contrast_set.py')
score_module = importlib.util.module_from_spec(spec); spec.loader.exec_module(score_module)
archived_metrics = json.loads((ARTIFACT_DIR / 'metrics.json').read_text())

MODELS, loaded_models = {}, {}
alias_to_mode = {'head': 'head_only', 'mlp': 'mlp_adapter', 'lcca': 'lcca_adapter'}

def make_score_fn(model):
    @torch.no_grad()
    def score_fn(passage, question, option):
        hidden_cpu, mask_cpu = feature_cache[(passage, question, option)]
        hidden, mask = hidden_cpu.to(DEVICE), mask_cpu.to(DEVICE)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            score = model(hidden, mask, adapter_enabled=True).reshape(-1)[0]
        return float(score.float().cpu())
    return score_fn

for alias, mode in alias_to_mode.items():
    for seed in SEEDS:
        checkpoint = safe_load(ARTIFACT_DIR / 'checkpoints' / f'{alias}_seed{seed}.pt')
        model = FrozenFeatureScorer(mode, hidden_size).to(DEVICE)
        if model.adapter is not None:
            model.adapter.load_state_dict(checkpoint['adapter_state_dict'], strict=True)
        model.scorer.load_state_dict(checkpoint['scorer_state_dict'], strict=True)
        model.eval()
        loaded_models[(alias, seed)] = model
        MODELS[(alias, seed)] = make_score_fn(model)

gpu_metrics = score_module.run(contrast_rows, MODELS, treatment='lcca', control='mlp',
                               out_path=str(RESULT_DIR / 'metrics_gpu_recheck.json'))

parity_rows, failures = [], []
for run_key, expected in archived_metrics['per_run'].items():
    actual = gpu_metrics['per_run'][run_key]
    checks = {
        'contrast_consistency': (expected['contrast_consistency'], actual['contrast_consistency']),
        'flip_rate': (expected['flip_rate'], actual['flip_rate']),
        'balanced_accuracy_o': (expected['balanced_accuracy']['o'], actual['balanced_accuracy']['o']),
        'balanced_accuracy_a': (expected['balanced_accuracy']['a'], actual['balanced_accuracy']['a']),
        'balanced_accuracy_b': (expected['balanced_accuracy']['b'], actual['balanced_accuracy']['b']),
    }
    for metric, (exp, got) in checks.items():
        ok = exp == got
        parity_rows.append({'run': run_key, 'metric': metric, 'archived': exp, 'gpu_recheck': got, 'pass': ok})
        if not ok: failures.append((run_key, metric, exp, got))

parity = pd.DataFrame(parity_rows)
parity.to_csv(RESULT_DIR / 'cuda_parity_audit.csv', index=False)
if failures:
    display(parity[~parity['pass']])
    raise RuntimeError(f'CUDA PARITY FAILED in {len(failures)} values. Bootstrap intentionally not run.')
print('CUDA PARITY PASS: all 45 archived summary values matched exactly.')

raw_rows = []
gold_by_uid = {row['uid']: row for row in contrast_rows}
for (alias, seed), fn in sorted(MODELS.items()):
    preds = score_module.score_arms(contrast_rows, fn, alias, seed)
    for pred in preds:
        gold = gold_by_uid[pred['uid']]['gold_b' if pred['arm'] == 'b' else 'gold_o']
        raw_rows.append({**pred, 'gold': gold, 'correct': int(pred['pred'] == gold)})
raw = pd.DataFrame(raw_rows)
raw.to_csv(RESULT_DIR / 'contrast_predictions_raw_cuda.csv', index=False)
print('Raw CUDA prediction rows:', len(raw))

In [ ]:
# 6) Paired item-cluster bootstrap and sensitivity analyses.
def exact_mcnemar_p(a_only, b_only):
    n = a_only + b_only
    if n == 0: return 1.0
    k = min(a_only, b_only)
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / (2 ** n))

wide0 = raw.pivot_table(index=['uid', 'model', 'seed'], columns='arm', values='correct', aggfunc='first').reset_index()
wide0['consistency'] = ((wide0['a'] == 1) & (wide0['b'] == 1)).astype(int)
wide = wide0.pivot_table(index='uid', columns=['model', 'seed'], values='consistency', aggfunc='first').sort_index()
uids = wide.index.to_numpy()
lcca = wide['lcca'][list(SEEDS)].to_numpy(float)
mlp = wide['mlp'][list(SEEDS)].to_numpy(float)
delta = (lcca - mlp).mean(axis=1) * 100
point = float(delta.mean())

N_BOOT = 100_000
rng = np.random.default_rng(20260831)
item_draws, twoway_draws = np.empty(N_BOOT), np.empty(N_BOOT)
for start in range(0, N_BOOT, 5000):
    count = min(5000, N_BOOT - start)
    item_idx = rng.integers(0, len(uids), size=(count, len(uids)))
    la, ma = lcca[item_idx], mlp[item_idx]
    item_draws[start:start+count] = (la - ma).mean(axis=(1, 2)) * 100
    seed_idx = rng.integers(0, len(SEEDS), size=(count, len(SEEDS)))[:, None, :]
    twoway_draws[start:start+count] = (
        np.take_along_axis(la, seed_idx, axis=2) - np.take_along_axis(ma, seed_idx, axis=2)
    ).mean(axis=(1, 2)) * 100

lcca_majority = (lcca.sum(axis=1) >= 2).astype(int)
mlp_majority = (mlp.sum(axis=1) >= 2).astype(int)
lcca_only = int(((lcca_majority == 1) & (mlp_majority == 0)).sum())
mlp_only = int(((lcca_majority == 0) & (mlp_majority == 1)).sum())

def ci(x): return [round(float(v), 6) for v in np.quantile(x, [0.025, 0.975])]
result = {
    'cuda_parity_passed': True,
    'gpu': torch.cuda.get_device_name(0),
    'amp_dtype': str(AMP_DTYPE),
    'analysis_unit': '81 contrast UID clusters with three paired seed repeats retained',
    'point_gain_percentage_points': round(point, 6),
    'per_seed_gain_percentage_points': {
        str(seed): round(float((lcca[:, i] - mlp[:, i]).mean() * 100), 6)
        for i, seed in enumerate(SEEDS)
    },
    'item_cluster_bootstrap': {
        'replicates': N_BOOT,
        'ci_95_percentage_points': ci(item_draws),
        'probability_gain_le_zero': round(float((item_draws <= 0).mean()), 6),
    },
    'two_way_item_seed_bootstrap_sensitivity': {
        'replicates': N_BOOT,
        'ci_95_percentage_points': ci(twoway_draws),
        'probability_gain_le_zero': round(float((twoway_draws <= 0).mean()), 6),
        'caveat': 'Only three seeds are available; seed-population uncertainty is coarsely estimated.',
    },
    'item_majority_mcnemar': {
        'lcca_only_correct_clusters': lcca_only,
        'mlp_only_correct_clusters': mlp_only,
        'exact_two_sided_p': round(exact_mcnemar_p(lcca_only, mlp_only), 8),
        'lcca_majority_consistency_percent': round(float(lcca_majority.mean() * 100), 6),
        'mlp_majority_consistency_percent': round(float(mlp_majority.mean() * 100), 6),
    },
}
(RESULT_DIR / 'clustered_bootstrap_cuda_results.json').write_text(json.dumps(result, indent=2), encoding='utf-8')

cluster_table = pd.DataFrame({
    'uid': uids,
    'lcca_consistency_seed_mean': lcca.mean(axis=1),
    'mlp_consistency_seed_mean': mlp.mean(axis=1),
    'gain_percentage_points': delta,
    'lcca_majority_consistent': lcca_majority,
    'mlp_majority_consistent': mlp_majority,
})
cluster_table.to_csv(RESULT_DIR / 'contrast_item_clusters_cuda.csv', index=False)

summary_md = f"""| Analysis | Estimate | 95% interval | Additional evidence |
|---|---:|---:|---:|
| Item-cluster bootstrap | {point:+.2f} | [{result['item_cluster_bootstrap']['ci_95_percentage_points'][0]:+.2f}, {result['item_cluster_bootstrap']['ci_95_percentage_points'][1]:+.2f}] | Pr(gain ≤ 0) = {result['item_cluster_bootstrap']['probability_gain_le_zero']:.3f} |
| Two-way item/seed bootstrap | {point:+.2f} | [{result['two_way_item_seed_bootstrap_sensitivity']['ci_95_percentage_points'][0]:+.2f}, {result['two_way_item_seed_bootstrap_sensitivity']['ci_95_percentage_points'][1]:+.2f}] | Pr(gain ≤ 0) = {result['two_way_item_seed_bootstrap_sensitivity']['probability_gain_le_zero']:.3f} |
| Item-majority McNemar | {result['item_majority_mcnemar']['lcca_majority_consistency_percent'] - result['item_majority_mcnemar']['mlp_majority_consistency_percent']:+.2f} | n/a | {lcca_only} vs. {mlp_only}; p = {result['item_majority_mcnemar']['exact_two_sided_p']:.3f} |
"""
(RESULT_DIR / 'clustered_bootstrap_cuda_table.md').write_text(summary_md, encoding='utf-8')
print(json.dumps(result, indent=2))
print('\n' + summary_md)

In [ ]:
# 7) Package all exact CUDA artifacts and download automatically.
run_info = {
    'base_model_directory': str(DRIVE_MODEL_DIR),
    'artifact_zip': str(ARTIFACT_ZIP),
    'gpu': torch.cuda.get_device_name(0),
    'amp_dtype': str(AMP_DTYPE),
    'transformers_version': __import__('transformers').__version__,
    'torch_version': torch.__version__,
    'cuda_parity_passed': True,
}
(RESULT_DIR / 'cuda_run_environment.json').write_text(json.dumps(run_info, indent=2), encoding='utf-8')
shutil.copy2(ARTIFACT_DIR / 'checkpoint_manifest.csv', RESULT_DIR / 'checkpoint_manifest.csv')
shutil.copy2(ARTIFACT_DIR / 'metrics.json', RESULT_DIR / 'metrics_archived.json')

zip_base = '/content/lcca_v9_clustered_bootstrap_cuda_parity_results'
zip_path = shutil.make_archive(zip_base, 'zip', RESULT_DIR)
print('Created:', zip_path)
files.download(zip_path)